# FinGPT vs GPT: Stock Movement Forecasting
### Applied AI in Data Analytics: Final Project

## Research Question

Does fine-tuning on financial data actually give you better stock movement predictions, or can a general-purpose LLM with good prompting match a specialist model given the same inputs?

## Approach

We compare two models on an identical task:
- **FinGPT**: LLaMA-2-7B fine-tuned with a LoRA adapter on Dow 30 stocks (the specialist)
- **GPT-4o**: a general-purpose LLM via OpenAI API (the generalist)

Both receive the same prompt (company profile + recent price movement + news headlines) and produce a directional forecast. We then evaluate against actual market movement.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
assert torch.cuda.is_available(), 'You need A100 GPU! Runtime → Change runtime type → A100.'

CUDA available: True
Device: NVIDIA A100-SXM4-80GB


In [ ]:
!pip install -q openai yfinance finnhub-python peft transformers accelerate bitsandbytes sentencepiece

In [ ]:
from datetime import datetime, timedelta

DOW30_TICKERS = [
    'AAPL',   # Technology
    'MSFT',   # Technology
    'JPM',    # Financials
    'GS',     # Financials
    'JNJ',    # Healthcare
    'WMT',    # Consumer Staples
    'KO',     # Consumer Staples
    'CVX',    # Energy
    'CAT',    # Industrials
    'DIS',    # Media/Entertainment
]


NON_DOW_TICKERS = [
    'NFLX',   # Streaming (S&P 500, not Dow)
    'TSLA',   # EVs (S&P 500, not Dow)
    'NVDA',   # Semiconductors (was recently added to Dow in 2024,
              # but FinGPT was trained before: so still OOD for the model)
    'META',   # Social media (S&P 500, not Dow)
    'SHOP',   # E-commerce (TSX-listed, even more OOD)
]

# Combine for the experiment; we'll tag each one so we can analyze by group
TICKERS = DOW30_TICKERS + NON_DOW_TICKERS
TICKER_GROUP = {t: 'Dow30' for t in DOW30_TICKERS}
TICKER_GROUP.update({t: 'Non-Dow' for t in NON_DOW_TICKERS})

# Experiment window
LOOKBACK_DAYS = 30    # How much history we show the model
FORECAST_DAYS = 7     # How far ahead we ask it to predict

END_DATE = datetime(2025, 9, 1)
START_DATE = END_DATE - timedelta(days=LOOKBACK_DAYS)
FORECAST_END = END_DATE + timedelta(days=FORECAST_DAYS)

# GPT-5.4
GPT_MODEL = 'gpt-5.4'
MAX_HEADLINES = 10

print(f'Experiment setup:')
print(f'  Total tickers: {len(TICKERS)} ({len(DOW30_TICKERS)} Dow30 + {len(NON_DOW_TICKERS)} Non-Dow)')
print(f'  Data window: {START_DATE.date()} to {END_DATE.date()}')
print(f'  Forecast window: {END_DATE.date()} to {FORECAST_END.date()}')
print(f'  Generalist model: {GPT_MODEL}')
print(f'  Specialist model: FinGPT (LLaMA-2-7B + Dow30 LoRA)')
print(f'\n  Dow 30 (in-distribution): {", ".join(DOW30_TICKERS)}')
print(f'  Non-Dow (out-of-distribution): {", ".join(NON_DOW_TICKERS)}')

Experiment setup:
  Total tickers: 15 (10 Dow30 + 5 Non-Dow)
  Data window: 2025-08-02 to 2025-09-01
  Forecast window: 2025-09-01 to 2025-09-08
  Generalist model: gpt-5.4
  Specialist model: FinGPT (LLaMA-2-7B + Dow30 LoRA)

  Dow 30 (in-distribution): AAPL, MSFT, JPM, GS, JNJ, WMT, KO, CVX, CAT, DIS
  Non-Dow (out-of-distribution): NFLX, TSLA, NVDA, META, SHOP


In [ ]:
import yfinance as yf
import finnhub

finnhub_client = finnhub.Client(api_key=os.environ['FINNHUB_API_KEY'])


def fetch_stock_prices(ticker, start_date, end_date):
    stock = yf.Ticker(ticker)
    history = stock.history(start=start_date, end=end_date)
    if history.empty:
        raise ValueError(f'No price data for {ticker}')
    start_price = round(history['Close'].iloc[0], 2)
    end_price = round(history['Close'].iloc[-1], 2)
    pct_change = round((end_price - start_price) / start_price * 100, 2)
    return {
        'start_price': start_price,
        'end_price': end_price,
        'pct_change': pct_change,
        'start_date': start_date.strftime('%Y-%m-%d'),
        'end_date': end_date.strftime('%Y-%m-%d'),
    }


def fetch_company_profile(ticker):
    profile = finnhub_client.company_profile2(symbol=ticker)
    if not profile:
        raise ValueError(f'No profile for {ticker}')
    return {
        'name': profile.get('name', ticker),
        'finnhubIndustry': profile.get('finnhubIndustry', 'Unknown'),
        'ipo': profile.get('ipo', 'Unknown'),
        'marketCapitalization': profile.get('marketCapitalization', 0),
        'currency': profile.get('currency', 'USD'),
        'shareOutstanding': profile.get('shareOutstanding', 0),
        'country': profile.get('country', 'US'),
        'ticker': ticker,
        'exchange': profile.get('exchange', 'Unknown'),
    }


def fetch_news_headlines(ticker, start_date, end_date, max_headlines=10):
    news = finnhub_client.company_news(
        ticker,
        _from=start_date.strftime('%Y-%m-%d'),
        to=end_date.strftime('%Y-%m-%d'),
    )
    seen, unique_news = set(), []
    for item in news:
        headline = item.get('headline', '').strip()
        if headline and headline not in seen:
            seen.add(headline)
            unique_news.append({
                'headline': headline,
                'summary': item.get('summary', '').strip()[:300],
            })
        if len(unique_news) >= max_headlines:
            break
    return unique_news


def fetch_future_price(ticker, forecast_start, forecast_end):
    stock = yf.Ticker(ticker)
    future = stock.history(start=forecast_start, end=forecast_end)
    if future.empty:
        return None
    future_start = round(future['Close'].iloc[0], 2)
    future_end = round(future['Close'].iloc[-1], 2)
    actual_change = round((future_end - future_start) / future_start * 100, 2)
    return {
        'forecast_start_price': future_start,
        'forecast_end_price': future_end,
        'actual_pct_change': actual_change,
        'actual_direction': 'UP' if actual_change > 0 else 'DOWN',
    }


print('Data fetching functions loaded.')

Data fetching functions loaded.


In [ ]:
# Pull all the data for all tickers
all_data = {}
for ticker in TICKERS:
    print(f'Fetching {ticker}...')
    all_data[ticker] = {
        'profile': fetch_company_profile(ticker),
        'prices': fetch_stock_prices(ticker, START_DATE, END_DATE),
        'news': fetch_news_headlines(ticker, START_DATE, END_DATE, MAX_HEADLINES),
        'ground_truth': fetch_future_price(ticker, END_DATE, FORECAST_END),
    }
print('\nDone!')

Fetching AAPL...
Fetching MSFT...
Fetching JPM...
Fetching GS...
Fetching JNJ...
Fetching WMT...
Fetching KO...
Fetching CVX...
Fetching CAT...
Fetching DIS...
Fetching NFLX...
Fetching TSLA...
Fetching NVDA...
Fetching META...
Fetching SHOP...

Done!


In [ ]:
SYSTEM_PROMPT = (
    "You are a seasoned stock market analyst. Your task is to list the "
    "positive developments and potential concerns for companies based on "
    "relevant news and basic financials from the past weeks, then provide "
    "an analysis and prediction for the companies' stock price movement "
    "for the upcoming week. Your answer format should be as follows:\n\n"
    "[Positive Developments]:\n1. ...\n\n"
    "[Potential Concerns]:\n1. ...\n\n"
    "[Prediction & Analysis]:\n..."
)


def build_user_prompt(data, forecast_days=7):
    profile, prices, news = data['profile'], data['prices'], data['news']

    intro = (
        f"[Company Introduction]:\n\n"
        f"{profile['name']} is a leading entity in the {profile['finnhubIndustry']} "
        f"sector. Incorporated and publicly traded since {profile['ipo']}, the "
        f"company has established its reputation as one of the key players in the "
        f"market. As of today, {profile['name']} has a market capitalization of "
        f"{profile['marketCapitalization']:.2f} in {profile['currency']}, with "
        f"{profile['shareOutstanding']:.2f} shares outstanding. "
        f"{profile['name']} operates primarily in {profile['country']}, trading "
        f"under the ticker {profile['ticker']} on the {profile['exchange']}.\n\n"
    )

    price_section = (
        f"From {prices['start_date']} to {prices['end_date']}, "
        f"{profile['name']}'s stock price went from ${prices['start_price']} "
        f"to ${prices['end_price']} ({prices['pct_change']:+.2f}%). "
        f"Company news during this period are listed below:\n\n"
    )

    news_section = ''
    for item in news:
        news_section += f"[Headline]: {item['headline']}\n"
        if item['summary']:
            news_section += f"[Summary]: {item['summary']}\n"
        news_section += '\n'

    end_dt = datetime.strptime(prices['end_date'], '%Y-%m-%d')
    forecast_start = (end_dt + timedelta(days=1)).strftime('%Y-%m-%d')
    forecast_end = (end_dt + timedelta(days=forecast_days)).strftime('%Y-%m-%d')
    period = f"{forecast_start} to {forecast_end}"

    instruction = (
        f"Based on all the information before {prices['end_date']}, let's first "
        f"analyze the positive developments and potential concerns for "
        f"{profile['ticker']}. Come up with 2-4 most important factors respectively "
        f"and keep them concise. Most factors should be inferred from company-related "
        f"news. Then make your prediction of the {profile['ticker']} stock price "
        f"movement for next week ({period}). Provide a summary analysis to support "
        f"your prediction."
    )

    return intro + price_section + news_section + instruction


# Preview one prompt to make sure it looks right
sample_prompt = build_user_prompt(all_data['AAPL'])
print(sample_prompt[:1500] + '...')

[Company Introduction]:

Apple Inc is a leading entity in the Technology sector. Incorporated and publicly traded since 1980-12-12, the company has established its reputation as one of the key players in the market. As of today, Apple Inc has a market capitalization of 3805351.56 in USD, with 14681.14 shares outstanding. Apple Inc operates primarily in US, trading under the ticker AAPL on the NASDAQ NMS - GLOBAL MARKET.

From 2025-08-02 to 2025-09-01, Apple Inc's stock price went from $202.73 to $231.7 (+14.29%). Company news during this period are listed below:

[Headline]: Apple's Expanding Game Content to Aid Services Growth: What's Ahead?
[Summary]: AAPL expands Arcade with new titles and a growing game hub as its Services revenues climb 13.3% in Q3 2025.

[Headline]: GOOGL Services Benefits From New AI-Powered Features: What's Ahead?
[Summary]: Alphabet's Google Services revenues surge on AI-driven Search, YouTube and Gemini innovations despite stiff competition.

[Headline]: Bill

In [ ]:
import time
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


def predict_with_gpt(user_prompt, model='gpt-4o'):
    start = time.time()
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0.7,
    )
    elapsed = time.time() - start
    return {
        'output': response.choices[0].message.content,
        'latency_seconds': round(elapsed, 2),
        'tokens_used': response.usage.prompt_tokens + response.usage.completion_tokens,
    }


gpt_results = {}
for ticker in TICKERS:
    print(f'Running GPT-5.4 on {ticker}...')
    prompt = build_user_prompt(all_data[ticker])
    gpt_results[ticker] = predict_with_gpt(prompt, model=GPT_MODEL)
    print(f"  Latency: {gpt_results[ticker]['latency_seconds']}s, Tokens: {gpt_results[ticker]['tokens_used']}")

print('\nGPT done!')

Running GPT-4o on AAPL...
  Latency: 15.19s, Tokens: 1263
Running GPT-4o on MSFT...
  Latency: 10.18s, Tokens: 1499
Running GPT-4o on JPM...
  Latency: 11.84s, Tokens: 1446
Running GPT-4o on GS...
  Latency: 12.22s, Tokens: 1444
Running GPT-4o on JNJ...
  Latency: 12.34s, Tokens: 1491
Running GPT-4o on WMT...
  Latency: 9.12s, Tokens: 1294
Running GPT-4o on KO...
  Latency: 11.89s, Tokens: 1358
Running GPT-4o on CVX...
  Latency: 10.64s, Tokens: 1360
Running GPT-4o on CAT...
  Latency: 10.77s, Tokens: 1458
Running GPT-4o on DIS...
  Latency: 11.53s, Tokens: 1357
Running GPT-4o on NFLX...
  Latency: 10.29s, Tokens: 1311
Running GPT-4o on TSLA...
  Latency: 11.33s, Tokens: 1413
Running GPT-4o on NVDA...
  Latency: 13.05s, Tokens: 1544
Running GPT-4o on META...
  Latency: 11.19s, Tokens: 1483
Running GPT-4o on SHOP...
  Latency: 11.1s, Tokens: 1482

GPT done!


In [ ]:

base_model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-2-7b-chat-hf',
    trust_remote_code=True,
    device_map='cuda:0',
    torch_dtype=torch.float16,
)

fingpt_model = PeftModel.from_pretrained(
    base_model, 'FinGPT/fingpt-forecaster_dow30_llama2-7b_lora'
)
fingpt_model = fingpt_model.eval()

fingpt_tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-chat-hf')
print('FinGPT loaded!')

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

FinGPT loaded!


In [ ]:
import re

B_INST, E_INST = '[INST]', '[/INST]'
B_SYS, E_SYS = '<<SYS>>\n', '\n<</SYS>>\n\n'


def predict_with_fingpt(user_prompt):
    final_prompt = B_INST + B_SYS + SYSTEM_PROMPT + E_SYS + user_prompt + E_INST
    inputs = fingpt_tokenizer(final_prompt, return_tensors='pt')
    inputs = {k: v.to(fingpt_model.device) for k, v in inputs.items()}

    start = time.time()
    res = fingpt_model.generate(
        **inputs,
        max_length=4096,
        do_sample=True,
        eos_token_id=fingpt_tokenizer.eos_token_id,
        use_cache=True,
    )
    elapsed = time.time() - start

    output = fingpt_tokenizer.decode(res[0], skip_special_tokens=True)
    answer = re.sub(r'.*\[/INST\]\s*', '', output, flags=re.DOTALL)
    return {'output': answer, 'latency_seconds': round(elapsed, 2)}


fingpt_results = {}
for ticker in TICKERS:
    print(f'Running FinGPT on {ticker}...')
    prompt = build_user_prompt(all_data[ticker])
    fingpt_results[ticker] = predict_with_fingpt(prompt)
    print(f"  Latency: {fingpt_results[ticker]['latency_seconds']}s")

print('\nFinGPT done!')

Running FinGPT on AAPL...
  Latency: 38.55s
Running FinGPT on MSFT...
  Latency: 43.96s
Running FinGPT on JPM...
  Latency: 38.85s
Running FinGPT on GS...
  Latency: 34.92s
Running FinGPT on JNJ...
  Latency: 44.08s
Running FinGPT on WMT...
  Latency: 29.49s
Running FinGPT on KO...
  Latency: 31.65s
Running FinGPT on CVX...
  Latency: 45.62s
Running FinGPT on CAT...
  Latency: 32.56s
Running FinGPT on DIS...
  Latency: 34.63s
Running FinGPT on NFLX...
  Latency: 45.19s
Running FinGPT on TSLA...
  Latency: 40.04s
Running FinGPT on NVDA...
  Latency: 42.95s
Running FinGPT on META...
  Latency: 41.22s
Running FinGPT on SHOP...
  Latency: 41.18s

FinGPT done!


In [ ]:
def extract_direction(output_text):
    match = re.search(r'\[Prediction.*?\](.*?)(?:\[|$)', output_text, re.DOTALL | re.IGNORECASE)
    text = match.group(1).lower() if match else output_text.lower()

    bullish = ['increase', 'rise', 'up', 'gain', 'bullish', 'positive', 'grow',
               'climb', 'surge', 'rally', 'higher']
    bearish = ['decrease', 'fall', 'down', 'loss', 'bearish', 'decline', 'drop',
               'plunge', 'lower', 'negative']

    b_count = sum(text.count(w) for w in bullish)
    s_count = sum(text.count(w) for w in bearish)
    if b_count > s_count:
        return 'UP'
    elif s_count > b_count:
        return 'DOWN'
    return 'UNCLEAR'


comparison = []
for ticker in TICKERS:
    gt = all_data[ticker]['ground_truth']
    if gt is None:
        continue

    gpt_dir = extract_direction(gpt_results[ticker]['output'])
    fingpt_dir = extract_direction(fingpt_results[ticker]['output'])

    comparison.append({
        'ticker': ticker,
        'group': TICKER_GROUP[ticker],
        'actual': gt['actual_direction'],
        'actual_pct': gt['actual_pct_change'],
        'gpt_pred': gpt_dir,
        'gpt_correct': gpt_dir == gt['actual_direction'],
        'gpt_latency': gpt_results[ticker]['latency_seconds'],
        'fingpt_pred': fingpt_dir,
        'fingpt_correct': fingpt_dir == gt['actual_direction'],
        'fingpt_latency': fingpt_results[ticker]['latency_seconds'],
        'agreement': gpt_dir == fingpt_dir,
    })


def print_results_table(rows, title):
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    print(f"{'Ticker':<8} {'Group':<10} {'Actual':<12} {'GPT':<8} {'FinGPT':<8} {'GPT✓':<7} {'FinGPT✓':<9} {'Agree':<6}")
    print('-' * 80)
    for c in rows:
        actual = f"{c['actual']} ({c['actual_pct']:+.1f}%)"
        print(f"{c['ticker']:<8} {c['group']:<10} {actual:<12} {c['gpt_pred']:<8} {c['fingpt_pred']:<8} "
              f"{'✓' if c['gpt_correct'] else '✗':<7} {'✓' if c['fingpt_correct'] else '✗':<9} "
              f"{'✓' if c['agreement'] else '✗':<6}")

    if rows:
        gpt_acc = sum(c['gpt_correct'] for c in rows) / len(rows) * 100
        fingpt_acc = sum(c['fingpt_correct'] for c in rows) / len(rows) * 100
        agreement = sum(c['agreement'] for c in rows) / len(rows) * 100
        print(f"\n  GPT-5.4 accuracy:  {gpt_acc:.1f}%")
        print(f"  FinGPT accuracy:   {fingpt_acc:.1f}%")
        print(f"  Agreement rate:    {agreement:.1f}%")


# Split by group for the key research question
dow30_results = [c for c in comparison if c['group'] == 'Dow30']
non_dow_results = [c for c in comparison if c['group'] == 'Non-Dow']

print_results_table(comparison, "OVERALL RESULTS")
print_results_table(dow30_results, "IN-DISTRIBUTION (Dow 30: FinGPT's training set)")
print_results_table(non_dow_results, "OUT-OF-DISTRIBUTION (Non-Dow stocks)")

# The key research finding
if dow30_results and non_dow_results:
    fingpt_dow_acc = sum(c['fingpt_correct'] for c in dow30_results) / len(dow30_results) * 100
    fingpt_non_dow_acc = sum(c['fingpt_correct'] for c in non_dow_results) / len(non_dow_results) * 100
    gpt_dow_acc = sum(c['gpt_correct'] for c in dow30_results) / len(dow30_results) * 100
    gpt_non_dow_acc = sum(c['gpt_correct'] for c in non_dow_results) / len(non_dow_results) * 100

    print(f"\n{'='*80}")
    print("KEY FINDING: Does FinGPT's specialist advantage hold outside its training set?")
    print(f"{'='*80}")
    print(f"\n  FinGPT: {fingpt_dow_acc:.1f}% on Dow30  →  {fingpt_non_dow_acc:.1f}% on Non-Dow  "
          f"(drop of {fingpt_dow_acc - fingpt_non_dow_acc:+.1f} pts)")
    print(f"  GPT-5.4: {gpt_dow_acc:.1f}% on Dow30  →  {gpt_non_dow_acc:.1f}% on Non-Dow  "
          f"(drop of {gpt_dow_acc - gpt_non_dow_acc:+.1f} pts)")


OVERALL RESULTS
Ticker   Group      Actual       GPT      FinGPT   GPT✓    FinGPT✓   Agree 
--------------------------------------------------------------------------------
AAPL     Dow30      UP (+4.3%)   UP       UP       ✓       ✓         ✓     
MSFT     Dow30      DOWN (-2.0%) DOWN     UP       ✓       ✗         ✗     
JPM      Dow30      DOWN (-1.8%) UP       UP       ✗       ✗         ✓     
GS       Dow30      UP (+1.0%)   UP       UP       ✓       ✓         ✓     
JNJ      Dow30      UP (+0.2%)   UP       UP       ✓       ✓         ✓     
WMT      Dow30      UP (+2.7%)   UP       UP       ✓       ✓         ✓     
KO       Dow30      DOWN (-1.6%) UP       UP       ✗       ✗         ✓     
CVX      Dow30      DOWN (-5.0%) UP       UP       ✗       ✗         ✓     
CAT      Dow30      UP (+1.7%)   DOWN     UNCLEAR  ✗       ✗         ✗     
DIS      Dow30      DOWN (-0.3%) UP       UP       ✗       ✗         ✓     
NFLX     Non-Dow    UP (+2.5%)   UP       UP       ✓       ✓      